In [ ]:
import pandas as pd
from src.utils.paths import RAW_DIR

csv_path = RAW_DIR / "classification" / "WA_Fn-UseC_-Telco-Customer-Churn.csv"
print(csv_path, csv_path.exists())

df = pd.read_csv(csv_path)
print(df.shape)

In [ ]:
print(f"Shape: {df.shape}\n")
print("Data Types:")
print(df.dtypes)
print("\nFirst 5 rows:")
display(df.head())

In [ ]:
print("Missing values per column:")
print(df.isna().sum()[df.isna().sum() > 0])

print(f"\nDuplicate rows: {df.duplicated().sum()}")

print(f"\ncustomerID uniqueness: {df['customerID'].nunique()} unique out of {len(df)} rows")

In [ ]:
target_col = 'Churn'
print("Target counts:")
print(df[target_col].value_counts())

print("\nTarget proportions:")
print(df[target_col].value_counts(normalize=True))

In [ ]:
# First, check the types pandas inferred
print(df.dtypes)

In [ ]:
# Observation: TotalCharges is stored as text. Keep the raw column, add a numeric one.
df['TotalCharges_num'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

In [ ]:
# Test the assumption instead of trusting it: blank TotalCharges <=> tenure == 0
print((df['TotalCharges_num'].isna() == (df['tenure'] == 0)).all())

In [ ]:
# Rows where the ratio is defined: a numeric TotalCharges and no division by zero
valid = df['TotalCharges_num'].notna() & (df['tenure'] > 0)
df_valid = df[valid].copy()
print(len(df_valid))

In [ ]:
# Hypothesis test: if TotalCharges ~ tenure * MonthlyCharges, the ratio is near 1
df_valid['charge_ratio'] = (
    df_valid['TotalCharges_num'] / (df_valid['tenure'] * df_valid['MonthlyCharges'])
)

print(df_valid['charge_ratio'].describe())
print(df_valid['charge_ratio'].quantile([0.01, 0.05, 0.95, 0.99]))
print(((df_valid['charge_ratio'] - 1).abs() <= 0.10).mean())

In [ ]:
# repr(...) on the last line shows the exact text stored in those cells, including any spaces, which print alone would hide.
missing_tc = df[df['TotalCharges_num'].isna()]

print(len(missing_tc))
print(missing_tc[['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges', 'Contract', 'Churn']])
print(repr(missing_tc['TotalCharges'].unique()))

In [ ]:
# Decision B: tenure == 0 -> TotalCharges = 0 (fixed domain rule, learns nothing from data)
rule_rows = df['tenure'] == 0
df.loc[rule_rows, 'TotalCharges_num'] = 0.0

print(rule_rows.sum())                       # rows the rule touched
print(df['TotalCharges_num'].isna().sum())   # expect 0
print(df['TotalCharges_num'].dtype)          # expect float64

In [ ]:
print(pd.crosstab(df['InternetService'], df['OnlineSecurity']))

In [ ]:
# Each add-on column and the parent column that should determine its "No ... service" value
checks = {
    'OnlineSecurity':   ('InternetService', 'No internet service'),
    'OnlineBackup':     ('InternetService', 'No internet service'),
    'DeviceProtection': ('InternetService', 'No internet service'),
    'TechSupport':      ('InternetService', 'No internet service'),
    'StreamingTV':      ('InternetService', 'No internet service'),
    'StreamingMovies':  ('InternetService', 'No internet service'),
    'MultipleLines':    ('PhoneService',    'No phone service'),
}

for col, (parent, label) in checks.items():
    same_rows = (df[col] == label) == (df[parent] == 'No')
    print(f"{col:18s} {same_rows.all()}")

In [ ]:
# target vs features
churn_flag = (df['Churn'] == 'Yes')
print(churn_flag.groupby(df['Contract']).agg(['mean', 'sum', 'count']).round(3))

In [ ]:
print(df.groupby('Churn')[['tenure', 'MonthlyCharges']].describe().T.round(2))
print(df.groupby('Contract')['tenure'].describe().round(1))

In [ ]:
churn_flag = (df['Churn'] == 'Yes')
print(churn_flag.groupby(df['InternetService']).agg(['mean', 'sum', 'count']).round(3))
print(df.groupby('InternetService')['MonthlyCharges'].median().round(2))

In [ ]:
print(df.groupby(['InternetService', 'Churn'])['MonthlyCharges']
        .agg(['median', 'count']).round(2))

## Leakage log: Telco Customer Churn

Status key:
- **no evidence found**: a test was run and found nothing
- **unverified**: not tested, or not testable from this file
- **needs documentation**: only the data source can answer
- **exclude from features**: not a leakage question; the column should not be a predictor

| Column | Why it might leak | What I'd need to know | Status |
|---|---|---|---|
| `customerID` | Not leakage. It is a unique key (7,043 values for 7,043 rows). One-hot encoded, it would let a flexible model memorize individual training rows instead of learning patterns. | Whether the ID has any structure (for example, an order tied to signup date). The format looks random, but this has not been tested. | **exclude from features**; keep only for record tracking |
| `tenure` | A churner's value stops at departure, while an active customer's value is "so far" and still growing. The two groups were measured differently. | Whether tenure was measured as of one snapshot date for everyone. | **needs documentation** |
| `TotalCharges` | It inherits any `tenure` problem, because `TotalCharges ≈ tenure × MonthlyCharges` (within ±10% for 94.6% of customers). It may also include charges accrued after the intended prediction date. | When the value was computed, and whether churners' totals stop at departure. | **needs documentation** |
| `Contract` | The strongest single association so far (42.7% churn for month-to-month vs 2.8% for two-year). A strong association is not evidence of leakage, but it justifies extra scrutiny. For churners the field might hold their final contract, not the one active at prediction time. | When the field was captured relative to the churn event. | **unverified** |
| `MonthlyCharges` | Might reflect a price or plan change made near departure. | Whether it is the charge as of the snapshot date. Tested: the median `charge_ratio` is 1.000 for both churners and non-churners (SDs 0.050 vs 0.051), so there is no group-level difference. The test has low power for a price change made shortly before departure, so it does not rule one out. | **unverified** |

In [ ]:
print(df_valid.groupby('Churn')['charge_ratio'].describe().round(3))

Whether it is the charge as of the snapshot date. Tested: the median charge_ratio is 1.000 for both churners and non-churners (SDs 0.050 vs 0.051), so there is no group-level difference. The test has low power for a price change made shortly before departure, so it does not rule one out.

## Modeling plan: Telco Customer Churn

### 1. Split strategy
- Hold out a stratified 20% test set (`random_state` fixed), about 1,409 rows and about 374 churners. It is not used for model selection, tuning, or threshold choice, and is evaluated once at the end.
- Select and compare models on the remaining 80% with **stratified 5-fold cross-validation**, using the same folds for every model.
- A random split is appropriate because each row is a distinct customer (`customerID` is unique), so there is no group structure. The file is a snapshot with no dates, so a time-based split is impossible. Results estimate performance on *similar* customers, not future ones.

### 2. Evaluation
- **Primary metric (model comparison):** average precision (PR-AUC) for `Churn = Yes`. It is threshold-free and suits a 26.5% positive class. Its no-skill baseline is the prevalence, about 0.265.
- **Secondary metric:** ROC-AUC, plus the confusion matrix, precision and recall at the chosen threshold.
- **Threshold:** chosen on cross-validated predictions from the training set, never on the test set. The choice depends on the relative cost of a retention offer versus a lost customer, **which is not known from this file**. Until that cost is known, report precision and recall across several thresholds.
- Accuracy is reported but not used for selection: always predicting "No" scores about 73.5%.

### 3. Baseline
- `DummyClassifier` predicting the majority class: accuracy about 73.5%, recall 0% for churn, precision undefined (reported as 0 with `zero_division=0`), PR-AUC about 0.265, ROC-AUC 0.5.
- Every other model must beat these numbers under the same cross-validation.

### 4. Preprocessing decisions
- `TotalCharges`: convert to numeric; rows with `tenure == 0` get 0.0. Fixed domain rule, learns nothing from the data. Assumption (blank means no billing yet) is undocumented in the source.
- Add-on columns: recode "No internet service" and "No phone service" to "No". Fixed rule; no information is lost as long as `InternetService` and `PhoneService` remain features.
- `customerID`: excluded from features; kept only to identify records.
- `SeniorCitizen`: pass through as a binary 0/1 indicator (not scaled). Other Yes/No columns are encoded to 0/1 consistently.
- Anything learned from data (scaling, encoding categories, any imputation) is fitted on training folds only, inside a scikit-learn `Pipeline` with `ColumnTransformer`.

### 5. First models
1. Dummy baseline.
2. Logistic regression (scaled numeric features, regularized): interpretable benchmark.
3. Decision tree with constrained depth: tests nonlinearity and interactions.

Random forest and gradient boosting are considered only after these results are understood. Tuning uses cross-validation on the training set only.

### 6. Known limitations and open items
- `tenure`, `TotalCharges`, `Contract`, and `MonthlyCharges` have unresolved leakage questions (see leakage log). Results may be optimistic if those fields were recorded under different conditions for churners.
- `TotalCharges` is approximately `tenure × MonthlyCharges`; the effect of this overlap on linear models is to be tested, not assumed.
- Churn definition and observation window are not specified in the file.